# Helm 03: dependencies, repositories and OCI

Charts consume other charts through `dependencies` in `Chart.yaml`; subchart values live under the dependency's name. Charts are distributed from HTTP repositories or as OCI artifacts.


In [ ]:
export HOME=/tmp
mkdir -p /source/work && cd /source/work && [ -d gitops-renderers ] || git clone -q --recurse-submodules https://github.com/cznewt/gitops-renderers.git
cd /source/work/gitops-renderers/examples/02-helm && export HOME=/tmp && helm dependency update web 2>&1 | tail -3 && cat web/Chart.lock && ls web/charts


In [ ]:
export HOME=/tmp
helm repo add podinfo https://stefanprodan.github.io/podinfo >/dev/null && helm search repo podinfo --versions | head -4


In [ ]:
export HOME=/tmp
helm show chart oci://ghcr.io/stefanprodan/charts/podinfo --version 6.15.0 | grep -E '^name:|^version:|^appVersion:' && cd /tmp && helm pull oci://ghcr.io/stefanprodan/charts/podinfo --version 6.15.0 && ls podinfo-*.tgz


Packaging is the release step: the `.tgz` is what a repository index or an OCI registry stores, and `helm template` renders it like a directory.


In [ ]:
cd /source/work/gitops-renderers/examples/02-helm
export HOME=/tmp
helm package web -d /tmp >/dev/null && helm template web /tmp/web-0.1.0.tgz -n web --skip-tests -f values-prod.yaml | grep -c '^kind:'


In [ ]:
cd /source/work/gitops-renderers/examples/02-helm
helm template web web -n web --skip-tests -f values-prod.yaml | yq 'select(.kind == "Deployment") | .metadata.name + ": " + .spec.template.spec.containers[0].image'


Try it: pin podinfo one minor version lower, update the lock, and see what changes in the output.
